# Pré-Processamento COCO 2017 para YOLO

Este notebook realiza o pré-processamento do dataset COCO 2017 para treinamento com YOLO.

**Instruções:**
1. Certifique-se que o arquivo `coco_bruto.tar.gz` foi extraído no diretório base
2. Execute as células na sequência numérica
3. Cada célula tem um propósito específico e comentários detalhados

## Célula 1: Configurações Iniciais

Rode esta célula para configurar os caminhos e as classes alvo

In [ ]:
import json
import shutil
from pathlib import Path
import time

# ==========================================
# 1. CONFIGURAÇÕES DO PROJETO (JUPYTER)
# ==========================================
# Pega o diretório atual onde este notebook está rodando
BASE_DIR = Path.cwd() 
COCO_DIR = BASE_DIR / "coco_2017"
ANNOTATIONS_FILE = COCO_DIR / "annotations" / "instances_train2017.json"
IMAGES_DIR = COCO_DIR / "train2017"

OUTPUT_DIR = BASE_DIR / "coco_dataset"
OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_LABELS = OUTPUT_DIR / "labels"

# Nossa meta perfeita de balanceamento
IMAGENS_POR_CLASSE = 3000

# Dicionário do COCO original
COCO_CLASSES = {
    1: 'person',
    2: 'bicycle',
    3: 'car',
    4: 'motorcycle',
    6: 'bus'
}

# Nosso mapeamento para o YOLO (Obrigatoriamente começando em 0)
YOLO_MAPPING = {
    1: 0, # person -> 0
    3: 1, # car -> 1
    4: 2, # motorcycle -> 2
    2: 3, # bicycle -> 3
    6: 4  # bus -> 4
}

print(f"✅ Diretório base configurado para: {BASE_DIR}")
print(f"🎯 Meta: {IMAGENS_POR_CLASSE} imagens por classe alvo.")

✅ Diretório base configurado para: /home/kevin/Documents/TCC/COCO 2017
🎯 Meta: 3000 imagens por classe alvo.


## Célula 2: Funções de Suporte (Limpeza e Matemática)

Rode esta célula para registrar as funções na memória do Jupyter

In [3]:
# ==========================================
# 2. FUNÇÕES DE SUPORTE
# ==========================================
def setup_directories():
    """Limpa execuções anteriores e cria a estrutura do YOLO."""
    if OUTPUT_DIR.exists():
        print("🧹 Limpando diretório antigo...")
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
    OUTPUT_LABELS.mkdir(parents=True, exist_ok=True)
    print(f"📁 Diretórios criados em: {OUTPUT_DIR}")

def convert_coco_to_yolo(bbox, img_width, img_height):
    """Converte [x_min, y_min, w, h] (COCO) para [x_center, y_center, norm_w, norm_h] (YOLO)."""
    x_min, y_min, w, h = bbox
    
    # Programação Defensiva: Garante que a caixa não vaze para fora da foto
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    w = min(w, img_width - x_min)
    h = min(h, img_height - y_min)

    x_center = (x_min + w / 2.0) / img_width
    y_center = (y_min + h / 2.0) / img_height
    norm_w = w / img_width
    norm_h = h / img_height

    return x_center, y_center, norm_w, norm_h

print("✅ Funções auxiliares carregadas!")

✅ Funções auxiliares carregadas!


## Célula 3: O Algoritmo de Mineração (O Cérebro do Script)

Rode esta célula. É aqui que o JSON é lido e a matemática de balanceamento e sobreposição de imagens acontece na RAM. Nenhuma foto é copiada ainda.

In [4]:
# ==========================================
# 3. LEITURA E BALANCEAMENTO LÓGICO
# ==========================================
start_time = time.time()
print("⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos dependendo da sua RAM)")

with open(ANNOTATIONS_FILE, 'r') as f:
    coco_data = json.load(f)
print("✅ JSON carregado na memória!")

# Mapeamento rápido de imagens
images_info = {img['id']: img for img in coco_data['images']}

# Agrupa anotações por imagem (filtrando multidões borradas e focando nas 4 classes)
annotations_by_image = {}
for ann in coco_data['annotations']:
    cat_id = ann['category_id']
    if cat_id in COCO_CLASSES and not ann.get('iscrowd', 0):
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)

# Dicionários de rastreamento para o nosso relatório final
selected_image_ids = set()
images_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}
objects_per_class = {cat_id: 0 for cat_id in COCO_CLASSES}

print(f"🎯 Iniciando a busca exata de {IMAGENS_POR_CLASSE} imagens por classe...")

# Abordagem de preenchimento (Greedy)
for target_cat_id, class_name in COCO_CLASSES.items():
    for img_id, anns in annotations_by_image.items():
        if images_per_class[target_cat_id] >= IMAGENS_POR_CLASSE:
            break # Bateu a meta dessa classe, vai pra próxima
            
        has_target_class = any(a['category_id'] == target_cat_id for a in anns)
        
        if has_target_class and img_id not in selected_image_ids:
            selected_image_ids.add(img_id)
            
            # Conta os objetos físicos dentro dessa foto e atualiza o rastreador de imagens
            classes_in_this_image = set()
            for a in anns:
                c_id = a['category_id']
                if c_id in COCO_CLASSES:
                    objects_per_class[c_id] += 1
                    classes_in_this_image.add(c_id)
                    
            for c_id in classes_in_this_image:
                images_per_class[c_id] += 1

print(f"\n🔥 Mineração concluída em {time.time() - start_time:.2f} segundos!")
print(f"Total de imagens únicas selecionadas para extração: {len(selected_image_ids)}")

⏳ Carregando o arquivo JSON do COCO... (Pode levar 1-2 minutos dependendo da sua RAM)
✅ JSON carregado na memória!
🎯 Iniciando a busca exata de 3000 imagens por classe...

🔥 Mineração concluída em 45.83 segundos!
Total de imagens únicas selecionadas para extração: 8534


## Célula 4: A Extração Física (Cópia das Imagens e Geração dos TXTs)

Rode esta célula. É ela que vai entrar na pasta gigante de 18 GB, copiar os milhares de arquivos selecionados e gerar as conversões do YOLO. Essa é a que mais demora.

In [5]:
# ==========================================
# 4. EXTRAÇÃO DE ARQUIVOS E ESCRITA DAS LABELS
# ==========================================
print("⚙️  Preparando diretórios de saída...")
setup_directories()

print("🚀 Iniciando a cópia física das imagens e geração dos .txt (Aguarde...)")
imgs_copiadas = 0

for img_id in selected_image_ids:
    img_data = images_info[img_id]
    file_name = img_data['file_name']
    img_width = img_data['width']
    img_height = img_data['height']
    
    src_img_path = IMAGES_DIR / file_name
    dst_img_path = OUTPUT_IMAGES / file_name
    
    if src_img_path.exists():
        shutil.copy2(src_img_path, dst_img_path)
        imgs_copiadas += 1
    else:
        continue # Ignora graciosamente se a foto não estiver no disco
        
    txt_name = Path(file_name).stem + ".txt"
    txt_path = OUTPUT_LABELS / txt_name
    
    with open(txt_path, 'w') as txt_file:
        for ann in annotations_by_image[img_id]:
            cat_id = ann['category_id']
            # Filtro defensivo: Só escreve no txt se for uma das nossas 4 classes alvo
            if cat_id in YOLO_MAPPING:
                yolo_class = YOLO_MAPPING[cat_id]
                x_c, y_c, w, h = convert_coco_to_yolo(ann['bbox'], img_width, img_height)
                txt_file.write(f"{yolo_class} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

print(f"✅ Fim da extração física! {imgs_copiadas} imagens copiadas com sucesso.")

⚙️  Preparando diretórios de saída...
📁 Diretórios criados em: /home/kevin/Documents/TCC/COCO 2017/coco_dataset
🚀 Iniciando a cópia física das imagens e geração dos .txt (Aguarde...)
✅ Fim da extração física! 8534 imagens copiadas com sucesso.


## Célula 5: A Geração do Relatório Tabular do TCC

Rode esta célula. Ela vai imprimir a tabela perfeita e limpa que você viu na sua versão anterior, pronta para você tirar print ou copiar para a monografia.

In [6]:
# ==========================================
# 5. RELATÓRIO FINAL
# ==========================================
print("\n" + "="*85)
print("📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (COCO BALANCEADO - TREINAMENTO DO ZERO)")
print("="*85)
print(f"{'Classe ID':<10} {'Nome da Classe':<15} {'Total de Objetos':<20} {'Total de Imagens':<20} {'Média de Objetos/Imagem'}")
print("-" * 85)

total_objetos_geral = 0
for cat_id, name in COCO_CLASSES.items():
    yolo_id = YOLO_MAPPING[cat_id]
    total_obj = objects_per_class[cat_id]
    total_img = images_per_class[cat_id]
    media = total_obj / total_img if total_img > 0 else 0
    total_objetos_geral += total_obj
    
    print(f"{yolo_id:<10} {name:<15} {total_obj:<20} {total_img:<20} {media:.2f}")

print("="*85)
print("\n📈 RESUMO GERAL")
print("="*85)
print(f"✓ Total de imagens processadas fisicamente: {imgs_copiadas}")
print(f"✓ Total de objetos detectados (Bounding Boxes): {total_objetos_geral}")

media_geral = total_objetos_geral / imgs_copiadas if imgs_copiadas > 0 else 0
print(f"✓ Média de objetos por imagem no Dataset: {media_geral:.2f}")
print(f"✓ Número de classes ativas: {len(COCO_CLASSES)}")
print("="*85)
print(f"\n🎉 Tudo pronto! Pode zipar a pasta '{OUTPUT_DIR.name}' e fazer upload no Drive!")


📊 ANÁLISE DE DISTRIBUIÇÃO DE CLASSES (COCO BALANCEADO - TREINAMENTO DO ZERO)
Classe ID  Nome da Classe  Total de Objetos     Total de Imagens     Média de Objetos/Imagem
-------------------------------------------------------------------------------------
0          person          33788                6800                 4.97
3          bicycle         6601                 3000                 2.20
1          car             13684                3726                 3.67
2          motorcycle      7114                 3000                 2.37

📈 RESUMO GERAL
✓ Total de imagens processadas fisicamente: 8534
✓ Total de objetos detectados (Bounding Boxes): 61187
✓ Média de objetos por imagem no Dataset: 7.17
✓ Número de classes ativas: 4

🎉 Tudo pronto! Pode zipar a pasta 'coco_dataset' e fazer upload no Drive!


## Próximos Passos

Assim que você terminar de baixar o gigante arquivo de 18 GB via terminal:

1. **Célula 1**: Configura todos os caminhos, classes e metas
2. **Célula 2**: Define as funções de transformação e limpeza
3. **Célula 3**: Lê o JSON e faz o balanceamento na RAM (não copia nada ainda)
4. **Célula 4**: Copia as imagens selecionadas e gera os arquivos `.txt` YOLO (pode levar tempo!)
5. **Célula 5**: Gera um relatório tabular bonito para sua monografia

**Resultado Final**: Um dataset YOLO balanceado e polido pronto para treinar YOLO, Faster R-CNN ou SSD! 🚀